# Building a RAG Chatbot Using LlamaIndex

LlamaIndex provides users with a simple way of creating a chatbot that works both with an LLM and data from a database. This combination is called Retrieval-Augmented Generation (RAG) and is used to give LLM's the ability to answer queries using data it was not trained on. This notebook will cover each step necessary to create a RAG chatbot using the Python SDK for Azure Cosmos DB for NoSQL. It demonstrates **7 search types** now available in `AzureCosmosDBNoSqlVectorSearch`:

| Search Type | Description |
|---|---|
| `vector` | Standard nearest-neighbour via `VectorDistance`, ordered by score |
| `vector_score_threshold` | Vector search filtered by a minimum similarity score |
| `full_text_search` | `FullTextContains` WHERE predicate — no `query_embedding` needed |
| `full_text_ranking` | `ORDER BY RANK FullTextScore(...)`, single or multi-field RRF — no `query_embedding` needed |
| `hybrid` | `ORDER BY RANK RRF(FullTextScore, VectorDistance)` |
| `hybrid_score_threshold` | Hybrid RRF + client-side score threshold filter |
| `weighted_hybrid_search` | Hybrid RRF + client-side per-component weight re-ranking |

Important Note: This sample requires you to have Azure Cosmos DB for NoSQL and Azure OpenAI accounts set up. To get started, visit:
-  [Azure Cosmos DB for NoSQL Python Quickstart](https://learn.microsoft.com/en-us/azure/cosmos-db/nosql/quickstart-python?pivots=devcontainer-codespace)
-  [Azure Cosmos DB for NoSQL Vector Search](https://learn.microsoft.com/en-us/azure/cosmos-db/nosql/vector-search)
-  [Azure OpenAI](https://learn.microsoft.com/en-us/azure/ai-services/openai/)

In [ ]:
%pip install llama-index
%pip install llama-index-embeddings-azure-openai
%pip install llama-index-llms-azure-openai
%pip install llama-index-vector-stores-azurecosmosnosql
%pip install gradio
%pip install python-dotenv

## Setup Azure OpenAI
Prior to beginning we need to set up the LLM and embedding model that will be used in the RAG chatbot.

In [ ]:
import os
import time
from dotenv import load_dotenv
from llama_index.llms.azure_openai import AzureOpenAI
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding

load_dotenv()

In [ ]:
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_API_ENDPOINT", "YOUR_AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY  = os.getenv("AZURE_OPENAI_API_KEY",      "YOUR_AZURE_OPENAI_API_KEY")
COSMOS_DB_URI         = os.getenv("COSMOS_DB_URI",              "YOUR_AZURE_COSMOSDB_URI")
COSMOS_DB_API_KEY     = os.getenv("COSMOS_DB_API_KEY",          "YOUR_AZURE_COSMOSDB_KEY")

llm = AzureOpenAI(
    model="gpt-35-turbo",
    deployment_name="gpt-35-turbo",
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2023-05-15",
)

embed_model = AzureOpenAIEmbedding(
    model="text-embedding-3-large",
    deployment_name="text-embedding-3-large",
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2023-05-15",
)

## Loading the data
The first step is to load the data using the LlamaIndex function SimpleDirectoryReader.

In [ ]:
import nest_asyncio
from llama_index.core import SimpleDirectoryReader
from llama_index.core import Document

nest_asyncio.apply()

In [ ]:
documents = SimpleDirectoryReader(input_dir=r"../../DataSet/CVPR2019/abstracts_pdf").load_data()

## Configure Global LlamaIndex Settings

In [ ]:
from llama_index.core import Settings

Settings.llm = llm
Settings.embed_model = embed_model

## Create the Vector Store and Index
The next step is to index the data loaded, this is done through vector embeddings. Prior to indexing it is important to initialize a Cosmos DB NoSql vector store where the embeddings will be stored.

In [ ]:
from azure.cosmos import CosmosClient, PartitionKey
from llama_index.vector_stores.azurecosmosnosql import (
    AzureCosmosDBNoSqlVectorSearch,
    AzureCosmosDBNoSqlVectorSearchType,
)
from llama_index.core import StorageContext
from llama_index.core import VectorStoreIndex
from llama_index.core.vector_stores import VectorStoreQuery

In [ ]:
# Create Cosmos DB client
client = CosmosClient(COSMOS_DB_URI, credential=COSMOS_DB_API_KEY)

# Vector indexing policy
indexing_policy = {
    "indexingMode": "consistent",
    "includedPaths": [{"path": "/*"}],
    "excludedPaths": [{"path": '/"_etag"/?'}],
    "vectorIndexes": [{"path": "/embedding", "type": "quantizedFlat"}],
}

vector_embedding_policy = {
    "vectorEmbeddings": [
        {
            "path": "/embedding",
            "dataType": "float32",
            "distanceFunction": "cosine",
            "dimensions": 3072,
        }
    ]
}

partition_key = PartitionKey(path="/id")
cosmos_container_properties = {"partition_key": partition_key}
cosmos_database_properties = {}

# Create vector store
store = AzureCosmosDBNoSqlVectorSearch(
    cosmos_client=client,
    vector_embedding_policy=vector_embedding_policy,
    indexing_policy=indexing_policy,
    cosmos_container_properties=cosmos_container_properties,
    cosmos_database_properties=cosmos_database_properties,
    create_container=True,
    database_name="rag_chatbot_example",
    container_name="vector_store",
)

storage_context = StorageContext.from_defaults(vector_store=store)

# Index the data
index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)
print("Vector index created successfully.")

## Full Text Search Store Setup
For full text search and hybrid search types, a separate container with `full_text_policy` and `fullTextIndexes` is required.

> **Note:** Full text search requires enabling the feature in your Cosmos DB for NoSQL account. See [Full Text Search in Azure Cosmos DB for NoSQL](https://learn.microsoft.com/en-us/azure/cosmos-db/nosql/full-text-search).

In [ ]:
# Indexing policy with both vector and full text indexes
fts_indexing_policy = {
    "indexingMode": "consistent",
    "includedPaths": [{"path": "/*"}],
    "excludedPaths": [{"path": '/"_etag"/?'}],
    "vectorIndexes": [{"path": "/embedding", "type": "quantizedFlat"}],
    "fullTextIndexes": [{"path": "/text"}],
}

# Full text policy enabling full text search on the 'text' field
full_text_policy = {
    "defaultLanguage": "en-US",
    "fullTextPaths": [{"path": "/text", "language": "en-US"}],
}

# Create full text search store
fts_store = AzureCosmosDBNoSqlVectorSearch(
    cosmos_client=client,
    vector_embedding_policy=vector_embedding_policy,
    indexing_policy=fts_indexing_policy,
    full_text_policy=full_text_policy,
    cosmos_container_properties=cosmos_container_properties,
    cosmos_database_properties=cosmos_database_properties,
    create_container=True,
    database_name="rag_chatbot_example",
    container_name="fts_store",
)

fts_storage_context = StorageContext.from_defaults(vector_store=fts_store)
fts_index = VectorStoreIndex.from_documents(documents, storage_context=fts_storage_context)
print("Full text search index created successfully.")

## Query the data
The following cells demonstrate all 7 search types. Each search type is accessed by passing `search_type` (an `AzureCosmosDBNoSqlVectorSearchType` enum value) to `store.query()`.

### 6a. Vector Search (default)
Standard nearest-neighbour search using `VectorDistance`, ordered by similarity score.

In [ ]:
# Generate an embedding for the query
query_text = "object detection in autonomous driving"
query_embedding = embed_model.get_text_embedding(query_text)

vector_query = VectorStoreQuery(
    query_embedding=query_embedding,
    similarity_top_k=5,
)

results = store.query(
    vector_query,
    search_type=AzureCosmosDBNoSqlVectorSearchType.VECTOR,
)

print(f"[vector] Returned {len(results.nodes)} results")
for node, score in zip(results.nodes, results.similarities):
    print(f"  score={score:.4f}  id={node.node_id}")

### 6b. Vector Search with Score Threshold
Same as vector search but filters results to only include nodes whose similarity score meets or exceeds a minimum `threshold`.

In [ ]:
results_threshold = store.query(
    vector_query,
    search_type=AzureCosmosDBNoSqlVectorSearchType.VECTOR_SCORE_THRESHOLD,
    threshold=0.7,
)

print(f"[vector_score_threshold] Returned {len(results_threshold.nodes)} results (threshold=0.7)")
for node, score in zip(results_threshold.nodes, results_threshold.similarities):
    print(f"  score={score:.4f}  id={node.node_id}")

### 6c. Full Text Search
Uses a `FullTextContains` WHERE predicate to find documents that contain specific keywords. **No `query_embedding` is required.**

In [ ]:
fts_query = VectorStoreQuery(similarity_top_k=5)  # No query_embedding needed

results_fts = fts_store.query(
    fts_query,
    search_type=AzureCosmosDBNoSqlVectorSearchType.FULL_TEXT_SEARCH,
    full_text_rank_filter=[
        {"search_field": "text", "search_text": "object detection"}
    ],
)

print(f"[full_text_search] Returned {len(results_fts.nodes)} results")
for node in results_fts.nodes:
    print(f"  id={node.node_id}  text_preview={node.get_content()[:80]}...")

### 6d. Full Text Ranking
Uses `ORDER BY RANK FullTextScore(...)` for BM25-style ranking. Supports single or multi-field RRF. **No `query_embedding` is required.**

In [ ]:
# Single-field full text ranking
results_ftr = fts_store.query(
    fts_query,
    search_type=AzureCosmosDBNoSqlVectorSearchType.FULL_TEXT_RANKING,
    full_text_rank_filter=[
        {"search_field": "text", "search_text": "neural network image recognition"}
    ],
)

print(f"[full_text_ranking] Returned {len(results_ftr.nodes)} results")
for node in results_ftr.nodes:
    print(f"  id={node.node_id}  text_preview={node.get_content()[:80]}...")

In [ ]:
# Multi-field full text ranking (RRF across multiple search terms/fields)
results_ftr_multi = fts_store.query(
    fts_query,
    search_type=AzureCosmosDBNoSqlVectorSearchType.FULL_TEXT_RANKING,
    full_text_rank_filter=[
        {"search_field": "text", "search_text": "convolutional neural network"},
        {"search_field": "text", "search_text": "image segmentation"},
    ],
)

print(f"[full_text_ranking multi-field] Returned {len(results_ftr_multi.nodes)} results")
for node in results_ftr_multi.nodes:
    print(f"  id={node.node_id}  text_preview={node.get_content()[:80]}...")

### 6e. Hybrid Search
Combines `FullTextScore` and `VectorDistance` using `ORDER BY RANK RRF(...)`.

In [ ]:
hybrid_query = VectorStoreQuery(
    query_embedding=query_embedding,
    similarity_top_k=5,
)

results_hybrid = fts_store.query(
    hybrid_query,
    search_type=AzureCosmosDBNoSqlVectorSearchType.HYBRID,
    full_text_rank_filter=[
        {"search_field": "text", "search_text": "object detection in autonomous driving"}
    ],
)

print(f"[hybrid] Returned {len(results_hybrid.nodes)} results")
for node, score in zip(results_hybrid.nodes, results_hybrid.similarities):
    print(f"  score={score:.4f}  id={node.node_id}")

### 6f. Hybrid Search with Score Threshold
Hybrid RRF search with an additional client-side score threshold filter.

In [ ]:
results_hybrid_threshold = fts_store.query(
    hybrid_query,
    search_type=AzureCosmosDBNoSqlVectorSearchType.HYBRID_SCORE_THRESHOLD,
    full_text_rank_filter=[
        {"search_field": "text", "search_text": "object detection in autonomous driving"}
    ],
    threshold=0.6,
)

print(f"[hybrid_score_threshold] Returned {len(results_hybrid_threshold.nodes)} results (threshold=0.6)")
for node, score in zip(results_hybrid_threshold.nodes, results_hybrid_threshold.similarities):
    print(f"  score={score:.4f}  id={node.node_id}")

### 6g. Weighted Hybrid Search
Hybrid RRF search with client-side per-component weight re-ranking. `weights=[full_text, vector]`. Applied after server returns results since CosmosDB SQL does not support weighted RRF natively.

In [ ]:
# weights[0] = weight for full text component
# weights[1] = weight for vector component
results_weighted = fts_store.query(
    hybrid_query,
    search_type=AzureCosmosDBNoSqlVectorSearchType.WEIGHTED_HYBRID_SEARCH,
    full_text_rank_filter=[
        {"search_field": "text", "search_text": "object detection in autonomous driving"}
    ],
    weights=[0.3, 0.7],  # Favour semantic (vector) over keyword (full text)
)

print(f"[weighted_hybrid_search] Returned {len(results_weighted.nodes)} results (weights=[0.3, 0.7])")
for node, score in zip(results_weighted.nodes, results_weighted.similarities):
    print(f"  score={score:.4f}  id={node.node_id}")

## Advanced Query Options

### WHERE Filter
Pass a raw CosmosDB SQL WHERE predicate via the `where` kwarg to pre-filter documents before the vector search.

In [ ]:
results_where = store.query(
    vector_query,
    search_type=AzureCosmosDBNoSqlVectorSearchType.VECTOR,
    where="c.metadata.file_type = 'pdf'",
)

print(f"[vector + where filter] Returned {len(results_where.nodes)} results")

### Offset / Limit (Pagination)
Use `offset_limit` for `OFFSET x LIMIT y` pagination. This replaces `TOP` for hybrid search types.

In [ ]:
results_paginated = store.query(
    vector_query,
    search_type=AzureCosmosDBNoSqlVectorSearchType.VECTOR,
    offset_limit="OFFSET 0 LIMIT 3",
)

print(f"[vector + offset_limit] Returned {len(results_paginated.nodes)} results (page 1, size 3)")

### Projection Mapping
Use `projection_mapping` to project specific Cosmos DB document fields into LlamaIndex node metadata keys.

In [ ]:
results_proj = store.query(
    vector_query,
    search_type=AzureCosmosDBNoSqlVectorSearchType.VECTOR,
    projection_mapping={"metadata": "node_metadata", "text": "content"},
)

print(f"[vector + projection_mapping] Returned {len(results_proj.nodes)} results")
if results_proj.nodes:
    print("  Metadata keys:", list(results_proj.nodes[0].metadata.keys()))

### Return with Vectors
Set `return_with_vectors=True` to include the raw embedding vectors in the returned nodes.

In [ ]:
results_with_vectors = store.query(
    vector_query,
    search_type=AzureCosmosDBNoSqlVectorSearchType.VECTOR,
    return_with_vectors=True,
)

print(f"[vector + return_with_vectors] Returned {len(results_with_vectors.nodes)} results")
if results_with_vectors.nodes:
    emb = results_with_vectors.nodes[0].embedding
    print(f"  Embedding dims: {len(emb) if emb else 'None'}")

## RAG Chatbot with Gradio UI
Build a simple RAG chatbot using the default vector query engine and a Gradio interface.

In [ ]:
import gradio as gr

In [ ]:
query_engine = index.as_query_engine()

def user_query(user_prompt, chat_history):
    # Create a timer to measure the time it takes to complete the request
    start_time = time.time()
    # Get LLM completion
    response = query_engine.query(user_prompt)
    # Stop the timer
    end_time = time.time()
    elapsed_time = round((end_time - start_time) * 1000, 2)
    print(response)
    # Append user message and response to chat history
    details = f"\n (Time: {elapsed_time}ms)"
    chat_history.append([user_prompt, str(response) + details])

    return gr.update(value=""), chat_history

In [ ]:
chat_history = []
with gr.Blocks() as demo:
    chatbot = gr.Chatbot(label="RAG Chatbot")

    msg = gr.Textbox(label="Ask me anything about the document!")
    clear = gr.Button("Clear")

    msg.submit(user_query, [msg, chatbot], [msg, chatbot], queue=False)

    clear.click(lambda: None, None, chatbot, queue=False)

# Launch the Gradio interface
demo.launch(debug=True)

In [ ]:
demo.close()
